# 06 · Synergy valuation and deal justification

Builds on the Stage 3 standalone DCF. Management guidance is quoted with its source; everything else is a labelled estimate.

* **Guidance:** ~$2.0bn run-rate cost synergies, starting in year 1 and fully realized by fiscal 2021 (Dec 2017 release and
  investor call; May 2018 proxy p.130).
* **Revenue synergies:** not quantified publicly. Excluded from all scenarios; the advisors' implied values are shown for reference only.
* **Integration costs:** not disclosed at either valuation date, so 1.0x run-rate is assumed. Disney's later ~$1.5bn severance
  estimate (May 2019) appears only in a hindsight row.
* **Discount rate:** the standalone WACC for each date, with +1pp and +2pp sensitivities. Synergy terminal growth is 1% (range 0–2%).
* **No double counting:** the standalone forecasts exclude synergies (the advisors combined the "Disney forecasts for RemainCo" and the
  "synergy estimates" separately, proxy p.145).

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_rows", 300)
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
res = subprocess.run([sys.executable, str(ROOT / "src" / "valuation" / "build_stage4.py")], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
print(res.stdout.strip()); assert res.returncode == 0, res.stdout + res.stderr
V = pd.read_csv(PROC / "verification_checks.csv")
SA = pd.read_csv(PROC / "synergy_assumptions.csv"); SO = pd.read_csv(PROC / "synergy_outputs.csv"); SO["v"] = pd.to_numeric(SO.value, errors="coerce")

In [ ]:
SA

## Synergy value by scenario

In [ ]:
m = ["gross_synergy_value", "pv_integration_costs", "net_synergy_value", "net_synergy_value_per_21cf_share", "standalone_plus_net_synergies_per_share"]
SO[(SO.section == "Synergy value") & SO.metric.isin(m)].pivot_table(index=["perspective", "case"], columns="metric", values="v").round(2)[m]

In [ ]:
cf = SO[SO.section == "Synergy cash flows"]
cf.pivot_table(index=["perspective", "case"], columns="metric", values="v").round(0)

In [ ]:
SO[SO.section.str.startswith("Reference")][["perspective", "case", "metric", "v", "notes"]]

## Sensitivities

In [ ]:
SO[SO.section == "Synergy sensitivity"].pivot_table(index="case", columns="perspective", values="v").round(0)

## Deal justification: how much of the premium over standalone value do synergies cover?

In [ ]:
dj = SO[SO.section == "Deal justification"]
dj.pivot_table(index="metric", columns=["perspective", "case"], values="v", aggfunc="first").round(2)

In [ ]:
v = V[V.check_type.str.startswith("synergy_")]; print(v.groupby(["check_type", "status"]).size()); assert (v.status == "pass").all()